In [ ]:
# Load libraries
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import resample
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.sparse import hstack
from sklearn.metrics import classification_report, roc_auc_score


# ================================================================================================
# ===================================== Preliminaries ============================================
# ================================================================================================

# Load data
train = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/train.csv', sep='|')
items = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/items.csv', sep='|')

df = train.merge(items, 
                   on='pid', 
                   how='left',
                   validate="m:1")


# ================================================================================================
# ====================================== Feature Engineering =====================================
# ================================================================================================


# We will perform the following feature engineering steps:
# 1. Sort the data by 'linedID and 'day' to maintain the temporal order for TimeSeriesSplit.
# 2. Take 10% of the whole dataset for model selection (The earliest 10%).
# 3. Create train/test temporal split
# 4. Apply TimeSeriesSplit
# 5. Drop highly correlated features
# 6. Drop irrelevant features
# 7. Handle missing values
# 8. Create new features
# 9. Balance the classes for 'order' prediction using undersampling.
# 10. Encode categorical variables and scale numerical features.
 

# ================================================================================================
# Sort Dataset
# ================================================================================================

# Sort the data by 'linedID and 'day' to maintain the temporal order for TimeSeriesSplit.
df = df.sort_values(["day", "lineID"])

# ================================================================================================
# Data Sampling
# ================================================================================================


# Take 10% of the whole dataset for model selection (The earliest 10%).
df_sample = df.iloc[:int(len(df) * 0.1)]


# ================================================================================================
# Feature Creation
# ================================================================================================


# Create a new column 'priceRatio'
df_sample['priceRatio'] = (df_sample['price'] / df_sample['competitorPrice'].replace(0, np.nan)).fillna(1.0)
df_sample['priceRatio'] = df_sample['priceRatio'].astype(float) # Convert the column to a float data type for cleanliness

# Create a new column 'unitsBought'
df_sample['unitsBought'] = (df_sample['revenue'] / df_sample['price']).round().fillna(0) # We fill any potential NaN values (e.g., from 0/0) with 0 and handle division by zero
df_sample['unitsBought'] = df_sample['unitsBought'].astype(int) # Convert the column to an integer data type for cleanliness

# Create a binary flag for missing 'competitorPrice'
df_sample['missingCompetitorPrice'] = df_sample['competitorPrice'].isnull().astype(int)

# Create weekDay feature from 'day' (assuming 'day' is a sequential day number)
df_sample['weekDay'] = (df_sample['day'] % 7).replace({0: 7})  # Map 0 to 7 for better interpretability


# ================================================================================================
# Train Test split
# ================================================================================================


# Let's define the targets. First task: predict 'order'.
X = df_sample.drop(columns=['order'])
y = df_sample['order']


# Split data (stratified, to preserve order/non-order ratio) ---
split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]


X_train.columns
X_test.columns


# ================================================================================================
# TimeSeriesSplit
# ================================================================================================

unique_days = sorted(X_train['day'].unique())

fold_results = []

for i in range(14, len(unique_days) - 7):

    # =============================================================================
    # Create rolling weekly split
    # =============================================================================

    train_days = unique_days[:i]
    val_days = unique_days[i:i+7]

    train_idx = X_train[X_train['day'].isin(train_days)].index
    val_idx = X_train[X_train['day'].isin(val_days)].index

    X_fold_train = X_train.loc[train_idx].copy()
    X_fold_val = X_train.loc[val_idx].copy()

    y_fold_train = y_train.loc[train_idx].copy()
    y_fold_val = y_train.loc[val_idx].copy()


# ================================================================================================
# Drop highly correlated features
# ================================================================================================


num_features = ['competitorPrice',
                'price',
                'revenue',
                'rrp',
                'priceRatio',
                'unitsBought']

cat_features = ['lineID',
                'day',
                'weekDay',
                'pid',
                'adFlag',
                'availability',
                'click',
                'basket',
                'manufacturer',
                'group',
                'content',
                'unit',
                'pharmForm',
                'genericProduct',
                'salesIndex',
                'category',
                'campaignIndex']

# Remove highly correlated features
correlation_matrix = X_fold_train[num_features].corr().abs()

upper = correlation_matrix.where(
    np.tri(correlation_matrix.shape[0], k=-1).astype(bool)
)

to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

X_fold_train.drop(columns=to_drop, inplace=True)
X_fold_val.drop(columns=to_drop, inplace=True)

# ================================================================================================
# Drop irrelevant columns
# ================================================================================================


drop_cols = [
    'click',
    'basket',
    'price',
    'revenue',
    'rrp',
    'lineID',
    'unitsBought',
    'day'
    ]

X_fold_train.drop(columns=drop_cols, inplace=True)
X_fold_val.drop(columns=drop_cols, inplace=True)


# ================================================================================================
# Handle missing values
# ================================================================================================


X_fold_train.fillna({
    'pharmForm': 'Missing',
    'category': 'Missing'
}, inplace=True)

X_fold_val.fillna({
    'pharmForm': 'Missing',
    'category': 'Missing'
}, inplace=True)


# ================================================================================================
# Balance the classes for 'order' prediction using undersampling.
# ================================================================================================


train_data = pd.concat([X_fold_train, y_fold_train], axis=1)

majority = train_data[train_data['order'] == 0]
minority = train_data[train_data['order'] == 1]

majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

train_balanced = pd.concat([
    majority_downsampled,
    minority
])

X_fold_train = train_balanced.drop(columns=['order'])
y_fold_train = train_balanced['order']


# ================================================================================================
# Encode categorical variables and scale numerical features.
# ================================================================================================


# Define remaining numerical and categorical features
cat_features = [
        'weekDay',
        'pid',
        'adFlag',
        'availability',
        'manufacturer',
        'group',
        'content',
        'unit',
        'pharmForm',
        'genericProduct',
        'salesIndex',
        'category',
        'campaignIndex'
    ]

remaining_num_features = [
    col for col in X_fold_train.columns
    if col not in cat_features
]

remaining_cat_features = [
    col for col in X_fold_train.columns
    if col in cat_features
]


# =============================================================================
# Convert categorical features to string
# =============================================================================


X_fold_train[remaining_cat_features] = (
    X_fold_train[remaining_cat_features].astype(str)
)

X_fold_val[remaining_cat_features] = (
    X_fold_val[remaining_cat_features].astype(str)
)


# =============================================================================
# Scale numerical features
# =============================================================================


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_fold_train[remaining_num_features]
)

X_val_scaled = scaler.transform(
    X_fold_val[remaining_num_features]
)


# =============================================================================
# Encode categorical features
# =============================================================================

encoder = OneHotEncoder(handle_unknown='ignore')

X_train_cat = encoder.fit_transform(
    X_fold_train[remaining_cat_features]
)

X_val_cat = encoder.transform(
    X_fold_val[remaining_cat_features]
)


# =============================================================================
# Combine features
# =============================================================================

X_train_final = hstack([
    X_train_scaled,
    X_train_cat
])

X_val_final = hstack([
    X_val_scaled,
    X_val_cat
])


# ================================================================================================
# Verify the shapes and contents of the datasets before modeling
# ================================================================================================


print("Training set shape:", X_train_final.shape)
print("Validation set shape:", X_val_final.shape)